# CS570 — Project Deliverable 1: Data Loading & Exploratory Analysis
**Team:** Pentanet  
**Members:** AZATBEK ISMAILOV, FSEHAYE MEDHANIE ,MIR AHMAD ALI , MIR AHMAD ALI , RAMESH MANDAMANEDI , YUEXUAN LU  
**Date:** February 25, 2026


In [2]:
import os

# ── CHANGE THIS to where your ml-1m files are ──────────────────
DATA_DIR = os.path.join(os.path.dirname(os.path.abspath('__file__')), '..', 'data', 'raw')
# ───────────────────────────────────────────────────────────────

RATINGS_PATH = os.path.join(DATA_DIR, 'ratings.dat')
USERS_PATH   = os.path.join(DATA_DIR, 'users.dat')
MOVIES_PATH  = os.path.join(DATA_DIR, 'movies.dat')

for path in [RATINGS_PATH, USERS_PATH, MOVIES_PATH]:
    status = 'found' if os.path.exists(path) else 'NOT FOUND'
    print(f'{status}: {path}')




found: d:\SFBU\Spring_semester_2026\CS570\Project-CS-570\notebooks\..\data\raw\ratings.dat
found: d:\SFBU\Spring_semester_2026\CS570\Project-CS-570\notebooks\..\data\raw\users.dat
found: d:\SFBU\Spring_semester_2026\CS570\Project-CS-570\notebooks\..\data\raw\movies.dat


In [3]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName('CS570-D1-MovieLens')
    .master('local[*]')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.driver.memory', '2g')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('ERROR')
print('Spark version:', spark.version)


Spark version: 3.5.0


## 1. Data Loading
Each file is loaded with an **explicit schema** using `StructType`/`StructField`. No `inferSchema=True`.


In [4]:
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, LongType, StringType, FloatType
)

RATINGS_SCHEMA = StructType([
    StructField('UserID',    IntegerType(), nullable=False),
    StructField('MovieID',   IntegerType(), nullable=False),
    StructField('Rating',    FloatType(),   nullable=False),
    StructField('Timestamp', LongType(),    nullable=False),
])

USERS_SCHEMA = StructType([
    StructField('UserID',     IntegerType(), nullable=False),
    StructField('Gender',     StringType(),  nullable=False),
    StructField('Age',        IntegerType(), nullable=False),
    StructField('Occupation', IntegerType(), nullable=False),
    StructField('ZipCode',    StringType(),  nullable=True),
])

MOVIES_SCHEMA = StructType([
    StructField('MovieID', IntegerType(), nullable=False),
    StructField('Title',   StringType(),  nullable=False),
    StructField('Genres',  StringType(),  nullable=False),
])
print('Schemas defined.')


Schemas defined.


In [8]:
# ratings.dat
ratings = spark.read.option('sep', '::').schema(RATINGS_SCHEMA).csv(RATINGS_PATH)
ratings.printSchema()
print('Row count:', ratings.count())
ratings.show(5)
ratings.columns

root
 |-- UserID: integer (nullable = true)
 |-- MovieID: integer (nullable = true)
 |-- Rating: float (nullable = true)
 |-- Timestamp: long (nullable = true)

Row count: 1000209
+------+-------+------+---------+
|UserID|MovieID|Rating|Timestamp|
+------+-------+------+---------+
|     1|   1193|   5.0|978300760|
|     1|    661|   3.0|978302109|
|     1|    914|   3.0|978301968|
|     1|   3408|   4.0|978300275|
|     1|   2355|   5.0|978824291|
+------+-------+------+---------+
only showing top 5 rows



['UserID', 'MovieID', 'Rating', 'Timestamp']

In [6]:
# users.dat
users = spark.read.option('sep', '::').schema(USERS_SCHEMA).csv(USERS_PATH)
users.printSchema()
print('Row count:', users.count())
users.show(5)
users.columns

root
 |-- UserID: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Occupation: integer (nullable = true)
 |-- ZipCode: string (nullable = true)

Row count: 6040
+------+------+---+----------+-------+
|UserID|Gender|Age|Occupation|ZipCode|
+------+------+---+----------+-------+
|     1|     F|  1|        10|  48067|
|     2|     M| 56|        16|  70072|
|     3|     M| 25|        15|  55117|
|     4|     M| 45|         7|  02460|
|     5|     M| 25|        20|  55455|
+------+------+---+----------+-------+
only showing top 5 rows



['UserID', 'Gender', 'Age', 'Occupation', 'ZipCode']

In [7]:
# movies.dat
movies = spark.read.option('sep', '::').schema(MOVIES_SCHEMA).csv(MOVIES_PATH)
movies.printSchema()
print('Row count:', movies.count())
movies.show(5)
movies.columns

root
 |-- MovieID: integer (nullable = true)
 |-- Title: string (nullable = true)
 |-- Genres: string (nullable = true)

Row count: 3883
+-------+--------------------+--------------------+
|MovieID|               Title|              Genres|
+-------+--------------------+--------------------+
|      1|    Toy Story (1995)|Animation|Childre...|
|      2|      Jumanji (1995)|Adventure|Childre...|
|      3|Grumpier Old Men ...|      Comedy|Romance|
|      4|Waiting to Exhale...|        Comedy|Drama|
|      5|Father of the Bri...|              Comedy|
+-------+--------------------+--------------------+
only showing top 5 rows



['MovieID', 'Title', 'Genres']

## 2. Join the Tables
`ratings` ↔ `users` on **UserID** · `ratings` ↔ `movies` on **MovieID** · both `inner` joins.


In [8]:
joined = (
    ratings
    .join(users,  on='UserID',  how='inner')
    .join(movies, on='MovieID', how='inner')
)

print('Row count:   ', joined.count())
print('Column count:', len(joined.columns))
joined.printSchema()
joined.show(5)


Row count:    1000209
Column count: 10
root
 |-- MovieID: integer (nullable = true)
 |-- UserID: integer (nullable = true)
 |-- Rating: float (nullable = true)
 |-- Timestamp: long (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Occupation: integer (nullable = true)
 |-- ZipCode: string (nullable = true)
 |-- Title: string (nullable = true)
 |-- Genres: string (nullable = true)

+-------+------+------+---------+------+---+----------+-------+--------------------+--------------------+
|MovieID|UserID|Rating|Timestamp|Gender|Age|Occupation|ZipCode|               Title|              Genres|
+-------+------+------+---------+------+---+----------+-------+--------------------+--------------------+
|   1193|     1|   5.0|978300760|     F|  1|        10|  48067|One Flew Over the...|               Drama|
|    661|     1|   3.0|978302109|     F|  1|        10|  48067|James and the Gia...|Animation|Childre...|
|    914|     1|   3.0|978301968|     F

## 3. Basic Statistics


In [9]:
joined.describe().show()


+-------+------------------+------------------+------------------+--------------------+-------+------------------+-----------------+------------------+--------------------+-------+
|summary|           MovieID|            UserID|            Rating|           Timestamp| Gender|               Age|       Occupation|           ZipCode|               Title| Genres|
+-------+------------------+------------------+------------------+--------------------+-------+------------------+-----------------+------------------+--------------------+-------+
|  count|           1000209|           1000209|           1000209|             1000209|1000209|           1000209|          1000209|           1000209|             1000209|1000209|
|   mean|1865.5398981612843| 3024.512347919285| 3.581564453029317| 9.722436954046655E8|   NULL| 29.73831369243828|8.036138447064564| 223239.8917114074|                NULL|   NULL|
| stddev|1096.0406894572482|1728.4126948999715|1.1171018453732606|1.2152558939916052E7|   NULL|

### Observations
[answer: What is the rating range? 
What does the mean tell you? What is unusual about Timestamp?]


## 4. EDA Questions


In [10]:
# A. Unique genres (explode pipe-separated values)
from pyspark.sql import functions as F

unique_genres = (
    joined
    .select(F.explode(F.split(F.col('Genres'), '\\|')).alias('genre'))
    .distinct()
    .count()
)
print(f'A. Unique individual genres: {unique_genres}')


A. Unique individual genres: 18


In [11]:
# B. Average rating — age group 25-34 (Age code = 25)
avg_25_34 = (
    joined
    .filter(F.col('Age') == 25)
    .agg(F.round(F.avg('Rating'), 2).alias('avg_rating'))
    .collect()[0]['avg_rating']
)
print(f'B. Average rating (25-34 age group): {avg_25_34}')


B. Average rating (25-34 age group): 3.55


In [12]:
# C. Movie with the most ratings
top = (
    joined
    .groupBy('MovieID', 'Title')
    .agg(F.count('*').alias('rating_count'))
    .orderBy(F.desc('rating_count'))
    .first()
)
print(f'C. Most rated movie : {top["Title"]}')
print(f'   Number of ratings: {top["rating_count"]}')


C. Most rated movie : American Beauty (1999)
   Number of ratings: 3428


## 5. Data Quality Observations


In [16]:
# Issue 1: Null values across all columns
joined.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in joined.columns
]).show()


+-------+------+------+---------+------+---+----------+-------+-----+------+
|MovieID|UserID|Rating|Timestamp|Gender|Age|Occupation|ZipCode|Title|Genres|
+-------+------+------+---------+------+---+----------+-------+-----+------+
|      0|     0|     0|        0|     0|  0|         0|      0|    0|     0|
+-------+------+------+---------+------+---+----------+-------+-----+------+



In [14]:
# Issue 2: Duplicate (UserID, MovieID) pairs
total    = joined.count()
distinct = joined.select('UserID', 'MovieID').distinct().count()
print(f'Total rows:       {total:,}')
print(f'Distinct pairs:   {distinct:,}')
print(f'Duplicates found: {total - distinct:,}')


Total rows:       1,000,209
Distinct pairs:   1,000,209
Duplicates found: 0


In [15]:
# Issue 3: Raw Timestamp — hard to read
joined.agg(F.min('Timestamp'), F.max('Timestamp')).show()
joined.select(F.from_unixtime('Timestamp').alias('readable_date')).show(5)


+--------------+--------------+
|min(Timestamp)|max(Timestamp)|
+--------------+--------------+
|     956703932|    1046454590|
+--------------+--------------+

+-------------------+
|      readable_date|
+-------------------+
|2000-12-31 14:12:40|
|2000-12-31 14:35:09|
|2000-12-31 14:32:48|
|2000-12-31 14:04:35|
|2001-01-06 15:38:11|
+-------------------+
only showing top 5 rows



### Issues Found
**Issue 1 —** [what did the null check reveal? which column?]
- Handle in D2: [your plan]

**Issue 2 —** [were there duplicates? what does that mean?]
- Handle in D2: [your plan]

**Issue 3 —** [what is wrong with raw timestamps?]
- Handle in D2: [your plan]


## 6. Contribution Statement

**AZATBEK ISMAILOV:** I contributed...

**FSEHAYE MEDHANIE:** I contributed...

**MIR AHMAD ALI:** I contributed...

**RAMESH MANDAMANEDI:** I contributed...

**YUEXUAN LU:** I contributed...
